# MOOCCubeX — Complete Google Drive Downloader and Colab Setup

This notebook:

1. Mounts Google Drive at `/content/drive`.
2. Stores everything under `/content/drive/MyDrive/DataCon/MOOCCubeX`.
3. Confirms that the Colab T4 GPU is available.
4. Downloads all available MOOCCubeX files with automatic resume and retry.
5. Skips the known broken `relations/concept-reply.json` URL.
6. Produces an inventory and safely previews large JSON files.

The complete dataset needs roughly **40–45 GB** of free Drive space. A free 15 GB Google Drive account cannot hold the complete dataset. The GPU accelerates later model training; it does not accelerate internet downloads.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
print('Google Drive mounted successfully.')


## Verify the T4 GPU

In Colab, select **Runtime → Change runtime type → T4 GPU** before running this cell.


In [ ]:
import subprocess
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. Select Runtime > Change runtime type > T4 GPU, then reconnect.')

device = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA version:', torch.version.cuda)
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
# Install tools used for resumable downloads and streaming JSON.
!apt-get -qq update
!apt-get -qq install -y aria2
!pip -q install ijson


In [ ]:
from pathlib import Path
import shutil

DRIVE_ROOT = Path('/content/drive/MyDrive/DataCon')
DATASET_ROOT = DRIVE_ROOT / 'MOOCCubeX'
LOG_ROOT = DRIVE_ROOT / 'logs'

for folder in [
    DATASET_ROOT / 'entities',
    DATASET_ROOT / 'relations',
    DATASET_ROOT / 'prerequisites',
    LOG_ROOT,
]:
    folder.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print('Dataset directory:', DATASET_ROOT)
print(f'Available Drive storage: {free_gb:.2f} GB')

if free_gb < 45:
    print('WARNING: The complete dataset may require 40–45 GB. Free more space or upgrade Drive storage.')


In [ ]:
BASE_URL = 'https://lfs.aminer.cn/misc/moocdata/data/mooccube2'

ALL_FILES = [
    'entities/reply.json',
    'entities/video.json',
    'entities/comment.json',
    'entities/course.json',
    'entities/other.json',
    'entities/paper.json',
    'entities/problem.json',
    'entities/school.json',
    'entities/teacher.json',
    'entities/user.json',
    'entities/concept.json',
    'relations/course-school.txt',
    'relations/course-teacher.txt',
    'relations/user-comment.txt',
    'relations/video_id-ccid.txt',
    'relations/comment-reply.txt',
    'relations/concept-other.txt',
    'relations/course-comment.txt',
    'relations/concept-video.txt',
    'relations/exercise-problem.txt',
    'relations/user-reply.txt',
    'relations/concept-comment.txt',
    'relations/concept-paper.txt',
    'relations/concept-problem.txt',
    # relations/concept-reply.json is excluded because its server URL returns 404.
    'relations/course-field.json',
    'relations/reply-reply.txt',
    'relations/user-problem.json',
    'relations/user-video.json',
    'relations/user-xiaomu.json',
    'prerequisites/psy.json',
    'prerequisites/cs.json',
    'prerequisites/math.json',
]

CORE_FILES = [
    'entities/video.json',
    'entities/user.json',
    'entities/course.json',
    'entities/concept.json',
    'relations/user-video.json',
    'relations/concept-video.txt',
    'relations/video_id-ccid.txt',
    'prerequisites/psy.json',
    'prerequisites/cs.json',
    'prerequisites/math.json',
]

# Keep True to download the complete available dataset.
# Change to False if you want only the files needed for the first recommender prototype.
DOWNLOAD_ALL = True
FILES_TO_DOWNLOAD = ALL_FILES if DOWNLOAD_ALL else CORE_FILES

print('Files selected:', len(FILES_TO_DOWNLOAD))


## Download the dataset

This cell writes directly to Google Drive. If Colab disconnects, reconnect, mount Drive, and run the cells again. `aria2c` will continue partial downloads using its `.aria2` control files.

Large files, particularly `relations/user-problem.json`, can take several hours. Keep the browser tab open while the cell is running.


In [ ]:
import subprocess
import time
from datetime import datetime

DOWNLOAD_LOG = LOG_ROOT / 'mooccubex_download.log'

def append_log(message):
    timestamp = datetime.now().isoformat(timespec='seconds')
    line = f'[{timestamp}] {message}'
    print(line)
    with DOWNLOAD_LOG.open('a', encoding='utf-8') as stream:
        stream.write(line + '\n')

def download_file(relative_path):
    destination = DATASET_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    url = f'{BASE_URL}/{relative_path}'

    append_log(f'Starting/resuming {relative_path}')
    command = [
        'aria2c',
        '--continue=true',
        '--max-connection-per-server=4',
        '--split=4',
        '--min-split-size=10M',
        '--max-tries=0',
        '--retry-wait=15',
        '--connect-timeout=60',
        '--timeout=120',
        '--file-allocation=none',
        '--auto-file-renaming=false',
        '--allow-overwrite=true',
        f'--dir={destination.parent}',
        f'--out={destination.name}',
        url,
    ]
    result = subprocess.run(command)
    if result.returncode == 0:
        append_log(f'Completed {relative_path} ({destination.stat().st_size / 1024**2:.2f} MB)')
    else:
        append_log(f'Interrupted/failed {relative_path}; run this cell again to resume')
    return result.returncode

failed = []
for number, relative_path in enumerate(FILES_TO_DOWNLOAD, start=1):
    print(f'\n[{number}/{len(FILES_TO_DOWNLOAD)}] {relative_path}')
    return_code = download_file(relative_path)
    if return_code != 0:
        failed.append(relative_path)

print('\nDownload pass finished.')
print('Files requiring another attempt:', failed if failed else 'None')
print('Log:', DOWNLOAD_LOG)


## Build and save the dataset inventory

Run this after each download session. Files with an accompanying `.aria2` file are still incomplete.


In [ ]:
import pandas as pd

rows = []
for relative_path in ALL_FILES:
    path = DATASET_ROOT / relative_path
    control_file = Path(str(path) + '.aria2')
    rows.append({
        'file': relative_path,
        'exists': path.exists(),
        'size_mb': round(path.stat().st_size / 1024**2, 2) if path.exists() else 0,
        'download_complete': path.exists() and not control_file.exists(),
    })

inventory = pd.DataFrame(rows)
inventory_path = DRIVE_ROOT / 'mooccubex_inventory.csv'
inventory.to_csv(inventory_path, index=False)

display(inventory)
print('Total downloaded:', round(inventory['size_mb'].sum() / 1024, 2), 'GB')
print('Inventory saved to:', inventory_path)


## Safely preview a large JSON file

The cell detects JSON Lines, a top-level JSON array, or a top-level JSON object. It reads only a few records and does not load the complete 3 GB file into memory.


In [ ]:
import json
import ijson

def preview_json(path, limit=3):
    path = Path(path)
    if not path.exists():
        print('File not found:', path)
        return []

    with path.open('rb') as stream:
        first = b''
        while not first:
            first = stream.read(1)
            if not first:
                return []
            first = first.strip()

    records = []
    if first == b'[':
        with path.open('rb') as stream:
            for record in ijson.items(stream, 'item'):
                records.append(record)
                if len(records) >= limit:
                    break
    elif first == b'{':
        # MOOCCubeX may use either JSON Lines or a top-level object.
        try:
            with path.open('r', encoding='utf-8') as stream:
                for line in stream:
                    line = line.strip()
                    if line:
                        records.append(json.loads(line))
                    if len(records) >= limit:
                        break
        except json.JSONDecodeError:
            records = []
            with path.open('rb') as stream:
                for key, value in ijson.kvitems(stream, ''):
                    records.append({key: value})
                    if len(records) >= limit:
                        break
    else:
        raise ValueError(f'Unsupported JSON structure in {path}')

    for index, record in enumerate(records, start=1):
        print(f'Record {index}:', record)
    return records

preview_json(DATASET_ROOT / 'relations/user-video.json', limit=3)


## Confirm the GPU for later model training

Use `device` when moving PyTorch models and tensors to the T4. Dataset parsing itself runs on the CPU.


In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
example_tensor = torch.randn(2048, 2048, device=device)
result = example_tensor @ example_tensor.T

print('Computation device:', result.device)
print('Allocated GPU memory:', round(torch.cuda.memory_allocated() / 1024**2, 2), 'MB')

del example_tensor, result
torch.cuda.empty_cache()
